In [21]:
import numpy as np
import pandas as pd

df = pd.read_excel("GSAF5.xls")
df.shape


(7125, 23)

In [22]:
def clean_location(df):
    df = df.copy()
    df["Location"] = (
        df["Location"]
        .astype("string")
        .str.replace(r"\([^)]*\)?", "", regex=True)
        .str.replace("?", "", regex=False)
        .str.replace(r"\s+", " ", regex=True)
        .str.replace(r"\s+,", ",", regex=True)
        .str.strip()
        .replace({"": pd.NA, "NaN": pd.NA, "nan": pd.NA, "N/A": pd.NA})  # exacto, no substring
        .replace({
            "300 miles east of Easter Island": "300 miles East of Easter Island",
            "Cheynes beach": "Cheynes Beach",
            "Kei River mouth": "Kei River Mouth",
            "Lauderdale-By-The-Sea, Broward County": "Lauderdale-by-the-Sea, Broward County",
            "Ponce De Leon Inlet, Volusia County": "Ponce de Leon Inlet, Volusia County",
            "South coast": "South Coast",
        })
        .fillna("Unknown")   # quita esta línea si prefieres mantener NA
    )
    return df

In [24]:
antes = df["Location"].astype("string")  # ejecútalo ANTES de clean_location
df = clean_location(df)
despues = df["Location"]

# --- Métricas ---
nulos_antes = antes.isna().sum()
nulos_despues = despues.isna().sum()
unknown_despues = (despues == "Unknown").sum()

print("Location - ANTES")
print("  Filas:", len(antes))
print("  Nulos:", nulos_antes)
print("  Valores únicos:", antes.nunique())

print("\nLocation - DESPUÉS")
print("  Filas:", len(despues))
print("  Nulos:", nulos_despues, f"({nulos_despues - nulos_antes:+d} vs antes)")
print("  'Unknown':", unknown_despues)
print("  Valores únicos:", despues.nunique(), f"({antes.nunique() - despues.nunique()} menos)")

print("\nComprobación de nulos:")
print("  Nulos ANTES == 'Unknown' DESPUÉS:", nulos_antes == unknown_despues)
print("  Diferencia ('NaN'/'N/A' en texto ya existentes):", unknown_despues - nulos_antes)

centinela = "<<NA>>"
cambiadas = antes.fillna(centinela) != despues.fillna(centinela)
print("\nFilas modificadas:", cambiadas.sum())

# --- Ejemplos ---
comparacion = pd.DataFrame({"antes": antes, "despues": despues})

print("\nHEAD (primeras 15 filas):")
print(comparacion.head(15).to_string())

print("\nMUESTRA ALEATORIA (15 filas):")
print(comparacion.sample(15, random_state=42).to_string())

print("\nEjemplos de cambios (excluyendo nulos -> Unknown):")
cambios_reales = cambiadas & antes.notna()
print(comparacion[cambios_reales].head(15).to_string())

# --- Distribución ---
print("\nTop 20 valores ANTES:")
print(antes.value_counts(dropna=False).head(20).to_string())

print("\nTop 20 valores DESPUÉS:")
print(despues.value_counts(dropna=False).head(20).to_string())

print("\nValores más raros ANTES (cola):")
print(antes.value_counts().tail(20).to_string())

Location - ANTES
  Filas: 7125
  Nulos: 0
  Valores únicos: 4570

Location - DESPUÉS
  Filas: 7125
  Nulos: 0 (+0 vs antes)
  'Unknown': 570
  Valores únicos: 4570 (0 menos)

Comprobación de nulos:
  Nulos ANTES == 'Unknown' DESPUÉS: False
  Diferencia ('NaN'/'N/A' en texto ya existentes): 570

Filas modificadas: 0

HEAD (primeras 15 filas):
                                        antes                                    despues
0                        Sorrento Beach Perth                       Sorrento Beach Perth
1   Off the coast of Perce Le Bilbo dive site  Off the coast of Perce Le Bilbo dive site
2                               Bimini Island                              Bimini Island
3                                   Geraldton                                  Geraldton
4                                    Honolulu                                   Honolulu
5                            New Smyrna Beach                           New Smyrna Beach
6                      30 miles o